Import Necessary Libraries

In [124]:
import pandas as pd
import re

Load Dataset

In [125]:
production = pd.read_csv("../outputs/production_crop_names.csv")

price = pd.read_csv("../outputs/price_commodity_names.csv")

print("Production Crops:", len(production))
print("Price Commodities:", len(price))

Production Crops: 124
Price Commodities: 389


Normalise data in Production Dataset and Price Dataset

In [126]:
def normalize(text):

    text = str(text).lower()

    text = text.replace("(", " ")
    text = text.replace(")", " ")

    text = re.sub(r"[/,-]", " ", text)

    text = re.sub(r"[^a-z0-9 ]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

Generate new columns as Normalised in both Price dataset and Production Dataset 

In [127]:
production["Normalized"] = production["Production_Crop"].apply(normalize)

price["Normalized"] = price["Price_Commodity"].apply(normalize)

Tokenisation Function

In [128]:
def tokens(text):
    return set(text.split())

Generate new column as Tokens in both dataset 

In [129]:
production["Tokens"] = production["Normalized"].apply(tokens)

price["Tokens"] = price["Normalized"].apply(tokens)

These are the excluded words which we are excluded from Processing

In [130]:
EXCLUDE_WORDS = {
    "dal",
    "atta",
    "flour",
    "oil",
    "bran",
    "powder",
    "ghee",
    "butter",
    "split"
}

These is the dictionary which we are using to search the items

In [131]:
SEARCH_TERMS = {

    "Blackgram": [
        "black gram",
        "urd beans"
    ],

    "Moong(Green Gram)": [
        "green gram",
        "moong"
    ],

    "Arhar/Tur": [
        "arhar",
        "red gram"
    ],

    "Bajra": [
        "bajra",
        "pearl millet"
    ],

    "Jowar": [
        "jowar",
        "sorghum"
    ],

    "Ragi": [
        "ragi",
        "finger millet"
    ],

    "Cowpea(Lobia)": [
        "cowpea",
        "lobia"
    ],

    "Horse-Gram": [
        "horse gram",
        "kulthi"
    ],

    "Sesamum": [
        "sesamum",
        "sesame",
        "gingelly"
    ],

    "Arecanut": [
        "arecanut",
        "betelnut",
        "supari"
    ],
    "Gram": [
        "bengal gram",
        "gram raw",
        "chana"
    ],
    "Cotton(Lint)": [
        "cotton"
    ]
}

These items are excluded

In [132]:
SPECIAL_EXCLUDE = {

    "Apple": {
        "Custard Apple (Sharifa)",
        "Pineapple"
    },

    "Rice": {
        "Broken Rice",
        "Beaten Rice"
    },

    "Ginger": {
        "Ginger (Dry)"
    }

}

Function which find possible Candidate in Price which matches with the Production crop


In [133]:
candidate_rows = []

for _, prod_row in production.iterrows():

    crop = prod_row["Production_Crop"]

    search_terms = SEARCH_TERMS.get(crop, [crop])

    for _, price_row in price.iterrows():

        commodity = price_row["Price_Commodity"]

        commodity_tokens = price_row["Tokens"]

        # Skip processed products
        if commodity_tokens.intersection(EXCLUDE_WORDS):
            continue

        # Skip crop-specific exclusions
        if crop in SPECIAL_EXCLUDE:
            if commodity in SPECIAL_EXCLUDE[crop]:
                continue

        # Check every search term
        matched = False

        matched_term = ""

        for term in search_terms:

            term_tokens = tokens(normalize(term))

            if term_tokens.issubset(commodity_tokens):

                matched = True
                matched_term = term
                break

        if matched:

            candidate_rows.append({

                "Production_Crop": crop,

                "Candidate_Commodity": commodity,

                "Matched_On": matched_term

            })

Create Dataframe

In [134]:
candidate_df = pd.DataFrame(candidate_rows)

candidate_df = candidate_df.drop_duplicates()

candidate_df = candidate_df.sort_values(
    ["Production_Crop", "Candidate_Commodity"]
)

candidate_df.head(100)

,Production_Crop,Candidate_Commodity,Matched_On
0,Apple,Apple,Apple
1,Arecanut,Arecanut (Betelnut/Supari),arecanut
2,Arhar/Tur,Arhar (Tur/Red Gram)(Whole),arhar
3,Arhar/Tur,Pegeon Pea (Arhar Fali),arhar
4,Arhar/Tur,Red Gram,red gram
...,...,...,...
95,Turnip,Turnip,Turnip
96,Varagu,Kodo Millet (Varagu),Varagu
97,Water Melon,Water Melon,Water Melon
98,Wheat,Wheat,Wheat


In [135]:
candidate_df[candidate_df["Production_Crop"]=="Apple"]

,Production_Crop,Candidate_Commodity,Matched_On
0,Apple,Apple,Apple


In [136]:
candidate_df[candidate_df["Production_Crop"]=="Moong(Green Gram)"]

,Production_Crop,Candidate_Commodity,Matched_On
59,Moong(Green Gram),Green Gram (Moong)(Whole),green gram


In [137]:
candidate_df[candidate_df["Production_Crop"]=="Blackgram"]

,Production_Crop,Candidate_Commodity,Matched_On
13,Blackgram,Black Gram (Urd Beans)(Whole),black gram


In [138]:
candidate_df[candidate_df["Production_Crop"]=="Arhar/Tur"]

,Production_Crop,Candidate_Commodity,Matched_On
2,Arhar/Tur,Arhar (Tur/Red Gram)(Whole),arhar
3,Arhar/Tur,Pegeon Pea (Arhar Fali),arhar
4,Arhar/Tur,Red Gram,red gram


In [139]:
candidate_df[candidate_df["Production_Crop"]=="Paddy"]

,Production_Crop,Candidate_Commodity,Matched_On
65,Paddy,Paddy (Basmati),Paddy
66,Paddy,Paddy (Common),Paddy
67,Paddy,Paddy (Dhan)(Basmati),Paddy
68,Paddy,Paddy (Dhan)(Common),Paddy


In [140]:
candidate_df[candidate_df["Production_Crop"]=="Gram"]

,Production_Crop,Candidate_Commodity,Matched_On
34,Gram,Bengal Gram (Gram)(Whole),bengal gram
35,Gram,Gram Raw (Chholia),gram raw
36,Gram,Kabuli Chana (Chickpeas-White),chana


In [141]:
candidate_df[candidate_df["Production_Crop"]=="Rice"]

,Production_Crop,Candidate_Commodity,Matched_On
77,Rice,Rice,Rice


In [142]:
candidate_df[candidate_df["Production_Crop"]=="Wheat"]

,Production_Crop,Candidate_Commodity,Matched_On
98,Wheat,Wheat,Wheat


In [143]:
candidate_df[candidate_df["Production_Crop"]=="Cotton(Lint)"]

,Production_Crop,Candidate_Commodity,Matched_On
26,Cotton(Lint),Cotton,cotton
27,Cotton(Lint),Cotton Seed,cotton


In [144]:
candidate_df[candidate_df["Production_Crop"]=="Sesamum"]

,Production_Crop,Candidate_Commodity,Matched_On
81,Sesamum,"Sesamum (Sesame,Gingelly,Til)",sesamum


In [145]:
candidate_df[candidate_df["Production_Crop"]=="Groundnut"]

,Production_Crop,Candidate_Commodity,Matched_On
39,Groundnut,Groundnut,Groundnut
40,Groundnut,Groundnut pods (raw),Groundnut


In [147]:
print(candidate_df.shape)

candidate_df.sort_values(
    ["Production_Crop", "Candidate_Commodity"]
)

(103, 3)


,Production_Crop,Candidate_Commodity,Matched_On
0,Apple,Apple,Apple
1,Arecanut,Arecanut (Betelnut/Supari),arecanut
2,Arhar/Tur,Arhar (Tur/Red Gram)(Whole),arhar
3,Arhar/Tur,Pegeon Pea (Arhar Fali),arhar
4,Arhar/Tur,Red Gram,red gram
...,...,...,...
98,Wheat,Wheat,Wheat
99,Yam,Elephant Yam (Suran),Yam
100,Yam,Elephant Yam (Suran)/Amorphophallus,Yam
101,Yam,Yam,Yam


In [146]:
candidate_df.to_csv(
    "../datasets/reference/crop_mapping_candidates.csv",
    index=False
)

In [148]:
final_mapping = candidate_df.rename(
    columns={
        "Candidate_Commodity": "Price_Commodity",
        "Production_Crop": "Standard_Crop"
    }
)

final_mapping = final_mapping[
    ["Price_Commodity", "Standard_Crop"]
]

final_mapping.to_csv(
    "../datasets/reference/crop_name_mapping.csv",
    index=False
)

final_mapping.head()

,Price_Commodity,Standard_Crop
0,Apple,Apple
1,Arecanut (Betelnut/Supari),Arecanut
2,Arhar (Tur/Red Gram)(Whole),Arhar/Tur
3,Pegeon Pea (Arhar Fali),Arhar/Tur
4,Red Gram,Arhar/Tur
